# Day 3: Modeling

**Data Science Mini Bootcamp 2026**  
Studi kasus: memprediksi `loan_status` pada Credit Risk Dataset.

Day 1 kita mengenali dataset melalui EDA. Day 2 kita menyiapkan data. Sekarang kita mencoba beberapa model, membaca hasilnya, lalu memilih satu model untuk dievaluasi pada test set.

**Yang perlu disiapkan:** tiga file CSV hasil preprocessing Day 2. File tersebut sudah tersedia di repository ini.

## Sebelum mulai

Pada akhir sesi, kamu akan bisa:

1. Menjelaskan fungsi train, validation, dan test set.
2. Melatih model dengan `.fit()` dan membuat prediksi dengan `.predict()`.
3. Membaca Precision, Recall, F1-score, dan Confusion Matrix.
4. Membandingkan model serta mencoba satu hyperparameter.

Kita menggunakan **Logistic Regression, Decision Tree, dan Random Forest**. Dummy Classifier dipakai sebagai baseline, bukan sebagai model utama.

## 1. Apa yang ingin diprediksi?

Kolom target kita adalah `loan_status`:

| Nilai | Arti |
|:--:|---|
| `0` | Non-default, pinjaman tidak gagal bayar |
| `1` | Default, pinjaman mengalami gagal bayar |

Ini adalah masalah **binary classification**. Dalam evaluasi, kita menyebut `1` sebagai **positive class**.

Contoh: jika orang yang sebenarnya default diprediksi non-default, itu disebut **False Negative**. Kesalahan tersebut penting diperhatikan karena bisa menyebabkan risiko kredit tidak terdeteksi.

> Studi kasus ini untuk latihan. Model belum boleh digunakan sebagai sistem persetujuan kredit nyata tanpa memeriksa ketersediaan fitur, bias, fairness, serta biaya kesalahan.

## 2. Siapkan library

Gunakan environment Python yang memiliki `pandas`, `numpy`, `matplotlib`, `scikit-learn`, dan `joblib`. Jika memakai Google Colab, library utama biasanya sudah tersedia.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report,
    ConfusionMatrixDisplay
)

## 3. Load hasil Preprocessing Day 2

Jalankan Day 2 sampai bagian **Export Processed Data**. File yang dipakai:

- `credit_train_processed.csv`
- `credit_validation_processed.csv`
- `credit_test_processed.csv`

Ketiganya sudah melewati encoding dan scaling di Day 2. **Tidak perlu split, encoding, atau scaling lagi** pada sesi ini.

In [ ]:
files_csv = [
    "credit_train_processed.csv",
    "credit_validation_processed.csv",
    "credit_test_processed.csv"
]
missing = [f for f in files_csv if not Path(f).exists()]
print("File belum ada:", missing or "Tidak ada")

In [ ]:
if missing:
    try:
        from google.colab import files
        print("Upload tiga CSV hasil Day 2:")
        files.upload()
    except ImportError:
        print("Simpan CSV di folder yang sama dengan notebook.")

In [ ]:
missing = [f for f in files_csv if not Path(f).exists()]
if missing:
    raise FileNotFoundError(f"File belum ditemukan: {missing}")
train_df, val_df, test_df = [pd.read_csv(f) for f in files_csv]

In [ ]:
for name, data in zip(["Train", "Validation", "Test"],
                      [train_df, val_df, test_df]):
    print(name, data.shape)
display(train_df.head(3))

### Pisahkan fitur dan target

`X` berisi kolom untuk prediksi. `y` berisi jawaban yang ingin dipelajari model. Jangan masukkan `loan_status` ke `X`, karena itu membuat model melihat jawabannya sendiri.

In [ ]:
TARGET = "loan_status"
X_train, y_train = train_df.drop(columns=TARGET), train_df[TARGET]
X_val, y_val = val_df.drop(columns=TARGET), val_df[TARGET]
X_test, y_test = test_df.drop(columns=TARGET), test_df[TARGET]
print("Jumlah fitur:", X_train.shape[1])

### Cek sebentar sebelum training

Day 2 menghasilkan 21 fitur numerik. Kita pastikan kolom antar-subset sama, tidak ada missing value, dan target hanya berisi `0` atau `1`. Jika ada error, periksa export Day 2.

In [ ]:
assert X_train.columns.equals(X_val.columns)
assert X_train.columns.equals(X_test.columns)
for X, y in [(X_train, y_train), (X_val, y_val), (X_test, y_test)]:
    assert set(y.unique()) <= {0, 1}
    assert X.select_dtypes(exclude="number").empty
    assert np.isfinite(X.to_numpy(dtype=float)).all()
assert set(y_train.unique()) == {0, 1}
print("Format data sudah sesuai.")

## 4. Train, validation, dan test dipakai untuk apa?

| Dataset | Perannya |
|---|---|
| **Train** | Model belajar menggunakan `.fit()` |
| **Validation** | Kita memilih model dan mengatur hyperparameter |
| **Test** | Kita memeriksa hasil akhir setelah model dipilih |

Aturannya: **jangan gunakan test set untuk memilih model**. Kalau kita terus mengganti model setelah melihat skor test, test tidak lagi menjadi ujian yang adil.

### Alur data dan pemilihan model

![Alur train, validation, dan test](https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBEWyJEYXRhIGhhc2lsIERheSAyIl0gLS0-IEFbIlRyYWluIl0KICAgIEQgLS0-IEJbIlZhbGlkYXRpb24iXQogICAgRCAtLT4gQ1siVGVzdCJdCiAgICBBIC0tPiBNWyJMYXRpaCBtb2RlbCJdCiAgICBNIC0tPiBTWyJCYW5kaW5na2FuIG1vZGVsIl0KICAgIEIgLS0-IFMKICAgIFMgLS0-IEVbIk1vZGVsIHRlcnBpbGloIl0KICAgIEUgLS0-IFRbIkV2YWx1YXNpIGFraGlyIl0KICAgIEMgLS0-IFQK?type=png&theme=neutral&bgColor=FFFFFF&width=900)

*Diagram Mermaid. [Lihat source](https://github.com/NafisNaufal/DSMBC-2026-Day3-Modeling/blob/main/diagrams/data_splits.mmd).*

Ketiga dataset sudah dipisahkan pada Day 2. **Train** dipakai untuk melatih model, **validation** untuk membandingkan model, dan **test** baru dipakai setelah model terpilih.


### Apakah kelasnya seimbang?

Lihat persentase kelas `0` dan `1` pada **train**. Perhatikan mengapa model yang selalu menebak kelas terbanyak mungkin terlihat bagus berdasarkan Accuracy.

In [ ]:
class_pct = y_train.value_counts(normalize=True).sort_index() * 100
class_pct.index = ["Non-default (0)", "Default (1)"]
display(class_pct.round(1).rename("Persentase (%)"))

## 5. Baseline: Dummy Classifier

Sebelum mencoba model lain, buat **baseline**. `most_frequent` selalu memilih kelas yang paling banyak muncul di train. Model ini bahkan tidak membaca pola pada fitur.

Pertanyaan untuk peserta: *Apa gunanya Accuracy tinggi jika model tidak pernah menemukan kasus default?*

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
pred_dummy = dummy.predict(X_val)
print("Accuracy:", round(accuracy_score(y_val, pred_dummy), 3))
print("F1 Default:", round(f1_score(y_val, pred_dummy), 3))

## 6. Cara membaca hasil classification

Untuk kelas Default (`1`), ada empat kemungkinan:

| Istilah | Maknanya |
|---|---|
| **TP** | Default, diprediksi default |
| **TN** | Non-default, diprediksi non-default |
| **FP** | Non-default, tetapi diprediksi default |
| **FN** | Default, tetapi diprediksi non-default |

**Precision:** dari yang diprediksi default, berapa yang benar?  
**Recall:** dari semua default sebenarnya, berapa yang ditemukan?  
**F1-score:** ringkasan yang menyeimbangkan Precision dan Recall.

Untuk sesi ini, kita akan memilih model berdasarkan **F1-score kelas Default** pada validation set. Dalam praktik nyata, biaya kesalahan juga perlu dipertimbangkan.

![Contoh Confusion Matrix](https://upload.wikimedia.org/wikipedia/commons/a/a6/Binary_confusion_matrix.png)

*Visual dari [Oritnk, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Binary_confusion_matrix.png), CC BY-SA 3.0. Gambar memakai label Yes/No, sedangkan kasus kita adalah Default/Non-default. Perhatikan bahwa baris menunjukkan kelas sebenarnya.*

In [ ]:
print(classification_report(
    y_val, pred_dummy, labels=[0, 1],
    target_names=["Non-default", "Default"],
    zero_division=0
))

## 7. Logistic Regression

Kita mulai dari model yang sederhana. **Logistic Regression** menghitung probabilitas untuk kelas `1` menggunakan kombinasi linear fitur. Meskipun namanya *regression*, model ini dapat dipakai untuk classification.

Hasil scaling dari Day 2 cocok untuk model ini.

### Dari fitur menjadi prediksi

![Alur Logistic Regression](https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBBWyJGaXR1ciBwaW5qYW1hbiJdIC0tPiBCWyJTa29yIGxpbmVhciJdCiAgICBCIC0tPiBDWyJTaWdtb2lkIl0KICAgIEMgLS0-IERbIlAoRGVmYXVsdCkiXQogICAgRCAtLT4gRXsicCA-PSAwLjU_In0KICAgIEUgLS0-fFlhfCBGWyJEZWZhdWx0ICgxKSJdCiAgICBFIC0tPnxUaWRha3wgR1siTm9uLWRlZmF1bHQgKDApIl0K?type=png&theme=neutral&bgColor=FFFFFF&width=900)

*Diagram Mermaid. [Lihat source](https://github.com/NafisNaufal/DSMBC-2026-Day3-Modeling/blob/main/diagrams/logistic_regression.mmd).*

Logistic Regression menghitung skor dari fitur, mengubahnya menjadi probabilitas dengan **sigmoid**, lalu memakai **threshold** untuk menentukan kelas. Dengan threshold `0.5`, probabilitas setidaknya `0.5` menghasilkan prediksi Default (`1`).


In [ ]:
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
pred_logreg = logreg.predict(X_val)
print("F1 Default:", round(f1_score(y_val, pred_logreg), 3))

## 8. Decision Tree

**Decision Tree** membuat aturan seperti *jika pendapatan sekian, lalu bagaimana kondisi pinjamannya?* Cabang-cabang tersebut memungkinkan model mempelajari hubungan nonlinear.

`max_depth=5` membatasi panjang jalur keputusan. Jika pohonnya terlalu dalam, model dapat terlalu menyesuaikan diri dengan train (**overfitting**).

![Contoh pohon keputusan pada Iris Dataset](https://upload.wikimedia.org/wikipedia/commons/a/a7/Classification_tree_on_iris_dataset.png)

*Visual: [Dvd8719, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Classification_tree_on_iris_dataset.png), CC BY-SA 4.0. Ini contoh pada Iris Dataset, bukan data kredit. Baca mulai dari kotak paling atas: setiap percabangan memeriksa suatu kondisi, lalu data masuk ke cabang yang sesuai.*

In [ ]:
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)
pred_tree = tree.predict(X_val)
print("F1 Default:", round(f1_score(y_val, pred_tree), 3))

## 9. Random Forest

Bagaimana kalau kita tidak hanya mengandalkan satu pohon? **Random Forest** melatih banyak Decision Tree dengan variasi sampel dan fitur, kemudian menggabungkan prediksinya. Pendekatan ini disebut **bagging**.

Modelnya lebih sulit dijelaskan daripada satu Decision Tree, tetapi sering menjadi baseline yang baik untuk data tabular.

![Random Forest dan bootstrap aggregation](https://upload.wikimedia.org/wikipedia/commons/e/e3/Random_Forest_Bagging_Illustration.png)

*Visual: [Harry585, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Random_Forest_Bagging_Illustration.png), CC BY-SA 4.0. Perhatikan tahap sampling, training beberapa pohon, lalu penggabungan prediksi.*

In [ ]:
forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)
forest.fit(X_train, y_train)

In [ ]:
pred_forest = forest.predict(X_val)
print("F1 Default:", round(f1_score(y_val, pred_forest), 3))

## 10. Coba tuning satu hyperparameter

Kita belum tahu apakah `max_depth=5` adalah pilihan terbaik untuk Decision Tree. Coba tiga nilai, lalu ukur **F1 di validation set**.

Kita sengaja memakai loop kecil agar efek satu parameter mudah terlihat. Untuk materi lanjut, proses ini bisa menggunakan `GridSearchCV`.

In [ ]:
depth_results = []
for depth in [3, 5, 8]:
    candidate = DecisionTreeClassifier(max_depth=depth, random_state=42)
    candidate.fit(X_train, y_train)
    score = f1_score(y_val, candidate.predict(X_val))
    depth_results.append((depth, score, candidate))

In [ ]:
depth_table = pd.DataFrame(
    [(depth, score) for depth, score, _ in depth_results],
    columns=["max_depth", "F1 Default"]
)
display(depth_table.round(3))

In [ ]:
best_depth, best_score, tuned_tree = max(
    depth_results, key=lambda result: result[1]
)
print("Kedalaman terpilih:", best_depth)
print("F1 validation:", round(best_score, 3))

## 11. Bandingkan model pada validation set

Kita pilih **satu** model menggunakan F1 kelas Default. Nilai F1 yang tinggi bukan jaminan bahwa model cocok untuk sistem kredit nyata, tetapi cukup untuk latihan pemilihan model.

Perhatikan juga Recall dan Precision. Dua model dengan F1 hampir sama dapat melakukan jenis kesalahan yang berbeda.

In [ ]:
def evaluate(model, X, y):
    pred = model.predict(X)
    return {
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1 Default": f1_score(y, pred, zero_division=0)
    }

In [ ]:
models = {
    "Dummy": dummy,
    "Logistic Regression": logreg,
    "Decision Tree": tree,
    "Random Forest": forest,
    "Decision Tree (tuned)": tuned_tree
}

### Bonus opsional: XGBoost

XGBoost adalah contoh **boosting**. Berbeda dari Random Forest yang membuat banyak pohon untuk kemudian digabungkan, boosting menambahkan model secara bertahap untuk memperbaiki prediksi sebelumnya.

Bagian ini bukan prasyarat. Jika baru pertama kali belajar modeling, **lewati saja** dan lanjutkan ke tabel perbandingan.

In [ ]:
COBA_XGBOOST = False  # Ganti True jika ingin mencoba bonus

In [ ]:
if COBA_XGBOOST:
    from xgboost import XGBClassifier
    xgb_model = XGBClassifier(
        n_estimators=120, max_depth=3, learning_rate=0.1,
        eval_metric="logloss", random_state=42, n_jobs=2
    )
    xgb_model.fit(X_train, y_train)
    models["XGBoost (bonus)"] = xgb_model

Kalau muncul `ModuleNotFoundError`, instal `xgboost` pada environment yang dipakai, lalu jalankan kembali cell bonus. Materi utama tidak membutuhkan package tersebut.

In [ ]:
results = pd.DataFrame({
    name: evaluate(model, X_val, y_val)
    for name, model in models.items()
}).T.sort_values("F1 Default", ascending=False)
display(results.round(3))

In [ ]:
results[["Precision", "Recall", "F1 Default"]].plot.bar(
    figsize=(9, 4), rot=25
)
plt.ylim(0, 1)
plt.title("Perbandingan Model pada Validation Set")
plt.tight_layout()
plt.show()

### Pilih satu model

Model teratas pada F1 validation menjadi pilihan kita. Setelah ini, kita **tidak memilih ulang model** berdasarkan hasil test.

In [ ]:
best_name = results.index[0]
best_model = models[best_name]
print("Model terpilih:", best_name)
print("F1 validation:", round(results.loc[best_name, "F1 Default"], 3))

### Apakah model overfitting?

Bandingkan F1 saat memprediksi train dan validation. Jika skor train jauh lebih tinggi, bisa jadi model terlalu mengikuti data train. Ini petunjuk awal, bukan diagnosis tunggal.

In [ ]:
train_f1 = f1_score(y_train, best_model.predict(X_train))
val_f1 = f1_score(y_val, best_model.predict(X_val))
print("F1 train:", round(train_f1, 3))
print("F1 validation:", round(val_f1, 3))

### Confusion Matrix milik model terpilih

Sekarang gunakan model yang sudah dipilih. Angka pada baris **Default** menunjukkan berapa kasus default yang berhasil ditemukan dan berapa yang terlewat.

In [ ]:
pred_val = best_model.predict(X_val)
ConfusionMatrixDisplay.from_predictions(
    y_val, pred_val,
    display_labels=["Non-default", "Default"],
    cmap="Blues", values_format="d"
)
plt.title("Confusion Matrix: Validation")
plt.show()

## 12. Mini Challenge: Prediksi dan threshold

**Challenge A.** Ambil tiga baris validation. Bandingkan kelas yang diprediksi model dengan probabilitas untuk Default.

Probabilitas `0.7` adalah estimasi model, bukan kepastian bahwa seseorang akan default. Probabilitas juga belum tentu terkalibrasi dengan baik.

In [ ]:
sample = X_val.head(3)
challenge_a = pd.DataFrame({
    "prediksi": best_model.predict(sample),
    "prob_default": best_model.predict_proba(sample)[:, 1],
    "aktual": y_val.head(3).to_numpy()
})
display(challenge_a.round(3))

**Challenge B.** Bandingkan threshold `0.35` dan `0.50` pada **Logistic Regression**. Bagaimana Precision dan Recall berubah?

Kita memakai Logistic Regression untuk contoh ini karena probabilitas prediksinya lebih bervariasi, sehingga perubahan threshold lebih mudah dilihat. Threshold lebih rendah membuat model lebih sering memprediksi Default (`1`). Percobaan ini **bukan pemilihan threshold final** dan tidak mengubah model yang sudah dipilih.

In [ ]:
proba_val = logreg.predict_proba(X_val)[:, 1]
for threshold in [0.35, 0.50]:
    pred = (proba_val >= threshold).astype(int)
    precision = precision_score(y_val, pred, zero_division=0)
    recall = recall_score(y_val, pred, zero_division=0)
    print(f"Threshold {threshold}: Precision={precision:.3f}, Recall={recall:.3f}")

## 13. Final test

Baru sekarang kita menggunakan `X_test` dan `y_test`. Hasil ini menjadi evaluasi akhir untuk model terpilih. Kita tidak akan mengganti model setelah melihatnya.

Jika metrik test lebih rendah dari validation, itu bisa terjadi karena data yang berbeda atau karena banyak percobaan selama pemilihan model.

In [ ]:
test_result = evaluate(best_model, X_test, y_test)
print("Model:", best_name)
print(pd.Series(test_result).round(3))

In [ ]:
print(classification_report(
    y_test, best_model.predict(X_test),
    target_names=["Non-default", "Default"],
    zero_division=0
))

## 14. Simpan model

Kita menyimpan model beserta **urutan nama fitur**. File ini hanya menerima data yang **sudah diproses dengan cara yang sama seperti Day 2**. File ini belum menjadi pipeline lengkap untuk input data mentah.

Dalam proyek nyata, semua proses preprocessing perlu disimpan juga agar prediksi baru konsisten.

In [ ]:
import joblib
artifact = {"model": best_model, "features": X_train.columns.tolist()}
joblib.dump(artifact, "dsmbc_model_processed.joblib")
print("Model tersimpan: dsmbc_model_processed.joblib")

In [ ]:
loaded = joblib.load("dsmbc_model_processed.joblib")
assert sample.columns.tolist() == loaded["features"]
print("Contoh prediksi setelah load:", loaded["model"].predict(sample).tolist())

## Diskusi penutup

Sebelum selesai, coba jawab secara lisan:

1. Kenapa Accuracy milik Dummy bisa terlihat tinggi?
2. Apa bedanya peran validation dan test?
3. Jika Recall meningkat tetapi Precision menurun, apa yang berubah pada jenis kesalahan model?
4. Kenapa file model ini belum cukup untuk memprediksi dari data pinjaman mentah?

**Yang perlu diingat:** training menggunakan train set, pemilihan menggunakan validation set, dan final evaluation menggunakan test set.

---
**Rujukan**

- [Day 1: EDA](https://github.com/clairedible/DataAnalyst_DSMBC)
- [Day 2: Preprocessing](https://github.com/khansafiryal/Preprocessing_DSMBC)
- [scikit-learn: Classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [scikit-learn: Decision Trees](https://scikit-learn.org/stable/modules/tree.html)

**Kredit visual:** gambar dari sumber publik ditautkan di bawah media. Diagram alur dibuat dengan Mermaid, dan source-nya tersedia di folder `diagrams/`. Visual eksternal membutuhkan koneksi internet. Grafik metrik yang dibuat oleh kode menggunakan data latihan sendiri.